# AI-Powered Weekly Operations Digest
### Regional Multi-Store Lead perspective

**Author:** Uzair

This notebook builds a prototype pipeline that turns four raw retail CSVs
(transactions, staffing shifts, returns, stores) into a weekly, per-store
metrics table, and then uses an LLM to generate a short, **fact-checked**
weekly digest for a Regional Multi-Store Lead.

**Chosen business perspective: Option B — Regional Multi-Store Lead.**
Reason: the most interesting patterns in this data are *comparative* —
one store has a staffing data gap in a specific week, and another has a
large, explainable revenue spike. A regional/comparative lens is the
natural fit for surfacing "which stores need attention" rather than a
single-store operational view.

**Pipeline:** Raw CSVs → Data Audit → Clean + Join → Weekly Metrics →
AI Digest (2 approaches) → Programmatic Claim Verification → Week-8 Holdout


In [1]:
import sys, os
sys.path.append(os.path.abspath("source_code"))

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

from data_audit import load_raw, audit_report, print_audit_summary
from cleaning import clean_all, build_weekly_store_table
from verifier import verify_claim, verify_claims, extract_claims_from_text


## Part 1 — Data Audit

Before touching the data, we run a set of standard checks: missing values,
duplicate records, invalid store references, invalid dates, unexpected
values, missing store/week combinations, and referential integrity.


In [2]:
stores, transactions, shifts, returns = load_raw()
report = audit_report(stores, transactions, shifts, returns)
print_audit_summary(report)


=== INVALID STORE REFERENCES (not in stores.csv) ===
  transactions: 35 records referencing ['S09']
  shifts: 15 records referencing ['S09']
  returns: 0 records referencing []

=== DUPLICATES ===
  transactions_full_row: 70
  transactions_by_id: 70
  shifts_full_row: 0
  returns_full_row: 0

=== MISSING VALUES ===
  transactions: {'promo_code': 8411}
  shifts: none
  returns: none

=== INVALID / UNPARSEABLE DATES ===
  transactions: 0
  shifts: 0
  returns: 0

=== VALUE RANGE CHECKS ===
  transactions_amount_min_max: (3.0, 289.43)
  transactions_nonpositive_amount: 0
  shifts_hours_min_max: (3.3, 9.8)
  shifts_nonpositive_hours: 0

=== MISSING STORE/WEEK COMBINATIONS (staffing) ===
  S04 / week 6: no staffing_shifts records


### Audit findings (Problem → Evidence → Decision → Reason)

**1. Invalid store references**
- *Problem:* `transactions.csv` and `staffing_shifts.csv` contain records for store `S09`, which does not exist in `stores.csv` (only S01–S05 are real stores).
- *Evidence:* 35 transactions and 15 shift records reference `S09`.
- *Decision:* Exclude all `S09` records from store-level aggregation.
- *Reason:* These rows cannot be reliably attributed to a real store; keeping them would corrupt every store-level rollup.

**2. Duplicate transactions**
- *Problem:* Exact duplicate rows exist in `transactions.csv`.
- *Evidence:* 70 rows are full-row duplicates (35 unique `transaction_id`s, each appearing exactly twice).
- *Decision:* Drop duplicates, keeping the first occurrence of each `transaction_id`.
- *Reason:* Counting both copies would double-count revenue and transaction volume for the affected stores/weeks.

**3. Missing store/week combination**
- *Problem:* Store `S04` has **zero** staffing-shift records for Week 6, despite having 223 transactions ($11,364.97 revenue) that same week.
- *Evidence:* Groupby(store, week) on `staffing_shifts.csv` shows no rows for `S04`/week 6.
- *Decision:* Represent `weekly_staffing_hours` and `sales_per_staffed_hour` as missing (`NaN`) for that store/week — not zero, and not an inferred/estimated value.
- *Reason:* A missing record is not the same as "zero hours worked." Dividing revenue by zero hours is undefined, and guessing a plausible number would itself be an unsupported claim — exactly what this assignment is designed to catch.

**4. Sparse `promo_code` field**
- *Problem:* `promo_code` is null in 8,411 of 8,541 transactions.
- *Evidence:* Only one non-null value exists across the whole dataset (`SPRING25`).
- *Decision:* Treat a blank `promo_code` as "no promotion applied," not as missing data requiring imputation.
- *Reason:* A promo code is inherently optional per transaction — this is expected structure, not a data-quality defect.

**5. Genuine business anomaly — retained, not "cleaned away"**
- *Problem:* Store `S03`'s Week-5 revenue ($23,516.04) is ~2.7× Week 4 ($8,671.07).
- *Evidence:* Traced to the `SPRING25` promo code (130 of 304 transactions that week) plus 4 temporary employees (`S03-TEMP01..04`) hired specifically for that week.
- *Decision:* Retain this data point as-is; it is not treated as an outlier to remove.
- *Reason:* It is a real, explainable business event. Removing it would hide a legitimate operational finding the digest should surface, not suppress.

*Note: per the assignment brief, this is not an exhaustive list of every possible anomaly — these are the issues judged most important to the resulting metrics.*


## Part 2 — Clean, Join, and Build the Weekly Dataset

Applying the decisions above, then joining all four files into one row
per (`store_id`, `week`).

**5 required metrics:**
- Weekly Revenue — sum of valid transaction amounts
- Weekly Transaction Count — count of valid transactions
- Weekly Return Rate — return amount ÷ weekly revenue
- Weekly Staffing Hours — sum of hours worked
- Sales per Staffed Hour — weekly revenue ÷ staffing hours

**3 additional metrics** (chosen for the Regional Multi-Store Lead lens — all support *comparing* stores):
- **Revenue WoW Change %** — week-over-week revenue trend per store
- **Online Sales Share %** — channel mix, useful for regional comparison
- **Revenue per Sq Ft** — normalizes revenue across stores of very different sizes (5,400–8,200 sqft), so small and large stores can be compared fairly


In [3]:
stores, transactions, shifts, returns = clean_all()
weekly = build_weekly_store_table(stores, transactions, shifts, returns)
weekly.round(2)


,store_id,region,week,weekly_revenue,weekly_transaction_count,weekly_return_rate,weekly_staffing_hours,sales_per_staffed_hour,revenue_wow_change_pct,online_sales_share_pct,revenue_per_sqft
0,S01,North,1,14315.48,255,0.02,319.9,44.75,NaN,32.55,1.75
1,S01,North,2,12756.93,254,0.02,310.3,41.11,-10.89,29.92,1.56
2,S01,North,3,12101.66,226,0.03,386.2,31.34,-5.14,33.19,1.48
3,S01,North,4,12738.10,229,0.04,337.3,37.76,5.26,32.75,1.55
4,S01,North,5,13699.01,257,0.03,367.9,37.24,7.54,33.07,1.67
5,S01,North,6,13643.96,261,0.02,312.0,43.73,-0.40,32.18,1.66
6,S01,North,7,14501.11,279,0.01,280.9,51.62,6.28,33.33,1.77
7,S01,North,8,14794.07,263,0.03,358.4,41.28,2.02,30.42,1.80
8,S02,South,1,10340.07,197,0.03,342.0,30.23,NaN,27.41,1.70
9,S02,South,2,11712.04,207,0.02,342.0,34.25,13.27,32.85,1.92


In [4]:
# The staffing gap and the revenue spike are both visible directly in the table:
print("Missing staffing data (Sales per Staffed Hour = N/A):")
display(weekly[weekly["sales_per_staffed_hour"].isnull()])

print("\nLargest week-over-week revenue jump:")
display(weekly.loc[[weekly["revenue_wow_change_pct"].idxmax()]])


Missing staffing data (Sales per Staffed Hour = N/A):


,store_id,region,week,weekly_revenue,weekly_transaction_count,weekly_return_rate,weekly_staffing_hours,sales_per_staffed_hour,revenue_wow_change_pct,online_sales_share_pct,revenue_per_sqft
29,S04,West,6,11364.97,223,0.009425,NaN,NaN,-8.221042,32.735426,1.556845



Largest week-over-week revenue jump:


,store_id,region,week,weekly_revenue,weekly_transaction_count,weekly_return_rate,weekly_staffing_hours,sales_per_staffed_hour,revenue_wow_change_pct,online_sales_share_pct,revenue_per_sqft
20,S03,East,5,23516.04,304,0.014516,416.8,56.420441,171.201132,22.697368,4.354822


## Part 3 & 4 — AI Weekly Digest: Two Approaches

We demonstrate both approaches on the same store/week — **Store S03, Week 5**
— because it contains a large, real, explainable spike, which is a good
stress test for whether the AI states facts accurately.

> **Before running this section:** create a `.env` file in this `Project/`
> folder containing one line: `ANTHROPIC_API_KEY=your_key_here`


### Approach 1 — Prompt-Based Grounding

We give the LLM the calculated metrics directly and instruct it to write
using only those numbers. No structured intermediate step, no automatic
verification during generation — we verify the free-text output afterwards.


In [5]:
from digest_approach1 import generate_digest_approach1

digest_text_1, current_1, previous_1 = generate_digest_approach1(weekly, "S03", 5)
print(digest_text_1)
import os
from dotenv import load_dotenv, find_dotenv

print("Current working directory:", os.getcwd())
print("Found .env at:", find_dotenv())

load_dotenv()
key = os.environ.get("ANTHROPIC_API_KEY")
print("Key loaded:", "YES - starts with " + key[:12] if key else "NO - not found")

For Week 5, Store S03 in the East region generated a Weekly Revenue of $2
Current working directory: d:\Intern_Assignment\Project
Found .env at: d:\Intern_Assignment\Project\.env
Key loaded: NO - not found


In [6]:
import os
from dotenv import load_dotenv, find_dotenv

print("Current working directory:", os.getcwd())
print("Found .env at:", find_dotenv())

load_dotenv()
key = os.environ.get("ANTHROPIC_API_KEY")
print("Key loaded:", "YES - starts with " + key[:12] if key else "NO - not found")

Current working directory: d:\Intern_Assignment\Project
Found .env at: d:\Intern_Assignment\Project\.env
Key loaded: NO - not found


In [7]:
# Verify Approach 1's free-text output: best-effort extraction of $ and % claims,
# then run each through the same verifier used for Approach 2.
extracted_claims = extract_claims_from_text(digest_text_1, store_id="S03", week=5)
print(f"Extracted {len(extracted_claims)} checkable claim(s) from the free text:\n")
for c in extracted_claims:
    print(" ", c)

print()
results_1 = verify_claims(extracted_claims, weekly)
for r in results_1:
    print(r["status"], "-", r["reason"], "| claimed:", r["claimed_value"], "actual:", r["actual_value"])


Extracted 1 checkable claim(s) from the free text:

  {'store_id': 'S03', 'week': 5, 'metric': 'weekly_revenue', 'claim_type': 'absolute', 'value': 2.0}

FAIL - Unsupported numerical claim - does not match computed value | claimed: 2.0 actual: 23516.04


### Approach 2 — Structured Generation + Verification

The LLM must first output a list of discrete, structured claims (JSON) —
not prose. Every claim is run through `verifier.py` **before** any of it
is allowed into the digest. Failed claims are dropped and flagged, not shown.


In [8]:
from digest_approach2 import generate_claims_approach2, build_verified_digest

claims_2, current_2, previous_2 = generate_claims_approach2(weekly, "S03", 5)
print("Raw structured claims from the LLM:")
for c in claims_2:
    print(" ", c)


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
digest_text_2, results_2 = build_verified_digest(claims_2, weekly, "S03", 5)
print(digest_text_2)
print()
print("Verification detail:")
for r in results_2:
    print(r["status"], "-", r["reason"], "| claimed:", r["claimed_value"], "actual:", r["actual_value"])


**Weekly Operations Digest — S03, Week 5**

- Weekly revenue for store S03 surged by 171.2% week-over-week in week 5.
- Sales per staffed hour increased significantly to $56.42 in week 5 from $23.09 in week 4.
- Weekly transaction count rose to 304 transactions in week 5.
- Online sales share accounted for 22.7% of total sales in week 5, down from 28.5% in week 4.
- The weekly return rate increased to 1.45% in week 5.

Verification detail:
PASS - Matches computed % change within tolerance | claimed: 171.20113204022113 actual: 171.20113204022113
PASS - Matches computed value within tolerance | claimed: 56.42044145873321 actual: 56.42044145873321
PASS - Matches computed value within tolerance | claimed: 304 actual: 304.0
PASS - Matches computed value within tolerance | claimed: 22.697368421052634 actual: 22.697368421052634
PASS - Matches computed value within tolerance | claimed: 0.014516049470914321 actual: 0.014516049470914321


### Comparing the two approaches

| Criterion | Approach 1 (Prompt-Based) | Approach 2 (Structured + Verification) |
|---|---|---|
| Accuracy | Depends entirely on the model following instructions | Every claim is checked against real numbers before display |
| Unsupported claims | Possible — nothing stops a wrong number from reaching the user | Caught and removed automatically |
| Ease of verification | Hard — requires parsing free text after the fact (fragile regex, ambiguous sentences) | Easy — claims already arrive in a checkable structured shape |
| Consistency | Wording and which facts get mentioned varies run to run | Same discrete facts checked every time; only wording of `text` field varies |
| Complexity | Simple — one prompt, one call | More moving parts — two LLM interaction points conceptually (claim generation + digest assembly), plus the verifier |

**Which approach would I choose and why:** Approach 2. Part 5 of this
assignment exists specifically because LLMs can state incorrect numbers
confidently. Approach 1 only lets us catch that *after the fact*, and only
as well as our free-text parser can find the numbers in the first place.
Approach 2 builds the check into the generation pipeline itself, so a wrong
number is dropped before a human ever sees it, and the final digest text is
assembled by our own code — not written freeform by the model — which makes
the output's provenance fully traceable.


## Part 5 — Claim Verification Function

See `source_code/verifier.py`. Core function: `verify_claim(claim, metrics_df)`.

A claim is a small structured dict:
```
{"store_id": "S03", "week": 5, "metric": "weekly_revenue",
 "claim_type": "absolute" | "change_percent", "value": <number>}
```
The verifier looks up the real value for that exact store/week/metric in
the cleaned metrics table (or recomputes the real % change from scratch for
`change_percent` claims) and compares it to the claimed value within a small
tolerance ($1 for absolute values, 0.5 percentage points for percentages).


## Part 10 — Required Verification Tests

Run against the exact example numbers given in the assignment brief
(Revenue = \$100,000, Previous Week = \$90,000), via `test_verifier.py`.


In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, "test_verifier.py"],
    cwd="source_code", capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


[OK ] Test 1: Correct number (Revenue = $100,000): expected=PASS got=PASS | claimed=100000 actual=100000.0 | Matches computed value within tolerance
[OK ] Test 2: Incorrect number (Revenue = $150,000): expected=FAIL got=FAIL | claimed=150000 actual=100000.0 | Unsupported numerical claim - does not match computed value
[OK ] Test 3: Correct percentage (+11.1%): expected=PASS got=PASS | claimed=11.1 actual=11.11111111111111 | Matches computed % change within tolerance
[OK ] Test 4: Incorrect percentage (+25%): expected=FAIL got=FAIL | claimed=25 actual=11.11111111111111 | Unsupported numerical claim - % change does not match
[OK ] Test 5: Wrong store (Store B's value claimed for Store A): expected=FAIL got=FAIL | claimed=40000 actual=100000.0 | Unsupported numerical claim - does not match computed value

All 5 required verification tests passed.



## Part 6 — Week-8 Holdout

Weeks 1–7 were used for all development and testing above. Now we run the
**final chosen approach (Approach 2)** on Week 8 for every store, exactly as
built — no logic changes made for Week 8.


In [ ]:
week8_results = {}
for store_id in sorted(stores["store_id"]):
    try:
        claims, cur, prev = generate_claims_approach2(weekly, store_id, 8)
        digest_text, verify_results = build_verified_digest(claims, weekly, store_id, 8)
        week8_results[store_id] = {"digest": digest_text, "verification": verify_results}
        print(digest_text)
        n_fail = sum(1 for r in verify_results if r["status"] == "FAIL")
        print(f"\n({len(verify_results)} claims generated, {n_fail} failed verification)")
        print("-" * 70)
    except Exception as e:
        print(f"{store_id}: ERROR - {e}")
        print("-" * 70)


**Weekly Operations Digest — S01, Week 8**

- Weekly revenue for store S01 increased by 2.02% week-over-week to $14,794.07.
- Weekly staffing hours saw a large increase of 27.59%, reaching 358.4 hours in week 8.
- Sales per staffed hour dropped by 20.04% to $41.28 due to higher staffing hours.
- The weekly return rate tripled from approximately 1.04% in week 7 to 3.12% in week 8.
- Total weekly transaction count declined to 263 transactions in week 8.

(5 claims generated, 0 failed verification)
----------------------------------------------------------------------
S02: ERROR - No JSON array found in model output. Raw response was:
[
  {
    "store_id": "S02",
    "week": 8,
    "metric": "weekly_revenue",
    "claim_type": "change_percent",
    "value": -5.69,
    "text": "Store S02 weekly revenue decreased by 5.69% week-over-week to $11,073.81 in week 8."
  },
  {
    "store_id": "S02",
    "week": 8,
    "metric": "weekly_staffing_hours",
    "claim_type": "change_percent",
    "val

### Week-8 holdout notes

*(Fill this in after actually running the cell above with your API key —
this is exactly the kind of thing the video needs to show and explain.)*

- What the system generated:
- Whether verification passed for all stores, or which ones had claims rejected:
- Anything unexpected (e.g. did S04's Week 8 staffing data exist this time?):
- Would you change anything after seeing this result?


## Part 7 — Findings

### Data
- **Records excluded:** 35 transactions + 15 shifts referencing the non-existent store `S09`; 70 duplicate transaction rows (35 unique IDs double-counted).
- **Missing data:** Store S04 has no staffing records for Week 6 — the only true store/week gap found. Represented as N/A throughout rather than guessed.
- **Important anomaly retained:** Store S03's Week 5 revenue spike (SPRING25 promo + temp staff) — a real event, not a data error, and a genuinely useful finding for a Regional Lead.

### Business
- S03 is the most volatile store week-to-week; its Week 5 promo produced the single largest revenue swing in the dataset (+171% WoW).
- S04's Week 6 is a reporting blind spot for staffing efficiency — worth flagging operationally even though sales that week were unremarkable.
- Revenue-per-sqft (an added metric) surfaces that store size alone doesn't explain performance — some smaller stores outperform larger ones on this basis.

### AI
- **Comparison:** Approach 2 (structured + verification) is more reliable and easier to audit than Approach 1 (prompt-only), at the cost of more implementation complexity.
- **Verification results:** All 5 required test cases pass (see Part 10). The verifier correctly caught a deliberately wrong value in earlier development testing.
- **Recommendation:** Approach 2, for the reasons given in the comparison table above.


## Limitations

- The free-text claim extractor for Approach 1 (`extract_claims_from_text`) uses simple regex patterns for `$` amounts and `increased/decreased by X%` phrasing. It will miss claims phrased differently, and cannot always disambiguate which store/week a sentence is about in a multi-store digest — this is itself evidence for preferring Approach 2.
- Tolerances in the verifier ($1, 0.5 percentage points) are reasonable defaults for this dataset's scale but were not tuned against a larger claim sample.
- This is a prototype: no persistence layer, no scheduling, no handling for weeks beyond the 8 provided.
